# Whisper 사투리 파인튜닝 (MacBook Pro M4 Pro / MPS 로컬 실행용)

Colab 버전(`whisper_dialect_finetuning.ipynb`)을 로컬(M4 Pro, 48GB 통합메모리) 환경에 맞게 수정한 버전입니다.

주요 변경점:
- Google Drive 마운트 제거 → 로컬 경로 사용
- CUDA 전제 코드 제거 → `mps` 디바이스 사용 (Apple Silicon GPU)
- 전체 오디오를 메모리에 미리 로드하던 방식 → `Dataset.__getitem__`에서 그때그때 로드(lazy loading)하는 방식으로 변경 (157GB 전체 데이터셋을 메모리에 다 올리면 무조건 OOM 남)
- 체크포인트 저장 빈도/보관 개수 조정 (디스크 용량 관리)
- 소규모 subset으로 먼저 파이프라인 검증할 수 있는 옵션 추가

**실행 전에 아래 경로만 본인 환경에 맞게 바꾸세요.**


In [1]:
import os

# AI-Hub 데이터 루트 (Training/Validation 상위 폴더)
BASE_DIR = os.path.expanduser(
    "~/Desktop/1_Project/0_충북대학교 석사 1학기/4_학회/260406_ICCAS/CalmChat/"
    "notebooks/139-1.중·노년층 한국어 방언 데이터 (강원도, 경상도)"
)

audio_dir = os.path.join(BASE_DIR, "Training", "01.원천데이터")
label_dir = os.path.join(BASE_DIR, "Training", "02.라벨링데이터")

model_out_dir = os.path.expanduser("~/dialect_data/model/whisper-dialect")
os.makedirs(model_out_dir, exist_ok=True)

# 하위 폴더(강원도/경상도, 1인발화/질문답변 등)까지 전부 재귀 탐색
audio_files = []
for root, _, files in os.walk(audio_dir):
    for f in files:
        if f.endswith(".wav"):
            audio_files.append(os.path.join(root, f))

label_files = []
for root, _, files in os.walk(label_dir):
    for f in files:
        if f.endswith(".json"):
            label_files.append(os.path.join(root, f))

print(f"오디오 파일 수: {len(audio_files)}")
print(f"라벨 파일 수: {len(label_files)}")
print(audio_files[:3])

오디오 파일 수: 303149
라벨 파일 수: 303149
['/Users/jm/Desktop/1_Project/0_충북대학교 석사 1학기/4_학회/260406_ICCAS/CalmChat/notebooks/139-1.중·노년층 한국어 방언 데이터 (강원도, 경상도)/Training/01.원천데이터/TS_02. 경상도_01. 1인발화 따라말하기/st_set1_collectorgs32_speakergs262_107_11.wav', '/Users/jm/Desktop/1_Project/0_충북대학교 석사 1학기/4_학회/260406_ICCAS/CalmChat/notebooks/139-1.중·노년층 한국어 방언 데이터 (강원도, 경상도)/Training/01.원천데이터/TS_02. 경상도_01. 1인발화 따라말하기/st_set1_collectorgs9_speakergs17_62_9.wav', '/Users/jm/Desktop/1_Project/0_충북대학교 석사 1학기/4_학회/260406_ICCAS/CalmChat/notebooks/139-1.중·노년층 한국어 방언 데이터 (강원도, 경상도)/Training/01.원천데이터/TS_02. 경상도_01. 1인발화 따라말하기/st_set3_collectorgs131_speakergs1302_48_9.wav']


In [2]:
!pip install -q -U transformers datasets librosa soundfile torch torchaudio evaluate
!pip uninstall -y torchvision


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [3]:
import torch

# MPS(Apple Silicon GPU) 미지원 연산은 자동으로 CPU로 폴백
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"사용 디바이스: {device}")


사용 디바이스: mps


In [4]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration

processor = WhisperProcessor.from_pretrained("openai/whisper-small")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")

print("모델 로드 완료")


/Users/jm/.pyenv/versions/3.13.3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 479/479 [00:00<00:00, 7465.59it/s]


모델 로드 완료


## 데이터 인덱싱 (오디오는 아직 로드하지 않음)

원본 코드는 여기서 오디오 파형(waveform)까지 전부 읽어 리스트에 쌓았습니다.
전체 데이터셋(157GB)에서 이 방식은 메모리 부족으로 죽기 때문에, **여기서는 경로/텍스트 쌍만** 모아두고
실제 오디오 로드는 학습 중 `Dataset.__getitem__`에서 그때그때 합니다.


In [ ]:
import json

def build_index(audio_paths, label_index):
    items = []
    skipped = []

    for audio_path in sorted(audio_paths):
        label_path = label_index.get(norm_key(audio_path))
        if not label_path:
            continue

        try:
            with open(label_path, 'r', encoding='utf-8') as f:
                label_data = json.load(f)
            text = label_data['transcription']['standard']
        except (json.JSONDecodeError, UnicodeDecodeError, KeyError) as e:
            skipped.append((label_path, str(e)))
            continue

        items.append({'audio_path': audio_path, 'text': text})

    print(f"인덱싱된 데이터 수: {len(items)}")
    print(f"스킵된(손상/구조 다른) 라벨 수: {len(skipped)}")
    for path, err in skipped[:5]:
        print(f" - {path}\n   {err}")

    return items
dataset_index = build_index(audio_files, label_index)
print(f"샘플 텍스트: {dataset_index[0]['text']}")

NameError: name 'label_index' is not defined

: 

In [ ]:
# 전체 데이터로 돌리기 전에, 먼저 일부만으로 파이프라인이 잘 도는지 확인하고 싶으면
# 아래 값을 정수로 바꾸세요 (예: 200). 전체로 돌리려면 None 유지.
DEBUG_SUBSET = None

if DEBUG_SUBSET:
    dataset_index = dataset_index[:DEBUG_SUBSET]
    print(f"디버그 모드: {len(dataset_index)}개 샘플만 사용")


In [ ]:
import librosa
from torch.utils.data import Dataset

class DialectDataset(Dataset):
    def __init__(self, items, processor):
        self.items = items
        self.processor = processor

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]

        speech, _ = librosa.load(item["audio_path"], sr=16000)
        inputs = self.processor(speech, sampling_rate=16000, return_tensors="pt")
        input_features = inputs.input_features[0]

        labels = self.processor.tokenizer(
            item["text"],
            return_tensors="pt",
            truncation=True,
            max_length=448,
        ).input_ids[0]

        return {"input_features": input_features, "labels": labels}
    
train_dataset = DialectDataset(dataset_index, processor)
print(f"데이터셋 크기: {len(train_dataset)}")


In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
import evaluate

def collate_fn(batch):
    input_features = torch.stack([item["input_features"] for item in batch])

    label_list = [item["labels"] for item in batch]
    max_len = max(l.size(0) for l in label_list)
    padded_labels = torch.full((len(label_list), max_len), -100, dtype=torch.long)
    for i, label in enumerate(label_list):
        padded_labels[i, : label.size(0)] = label

    return {"input_features": input_features, "labels": padded_labels}


## 학습 설정

- `fp16=False` 유지: MPS는 CUDA용 fp16 학습(GradScaler)을 지원하지 않습니다. fp32로 진행합니다.
- `per_device_train_batch_size=8`부터 시도해보세요. 48GB 통합메모리면 여유가 있는 편이지만,
  `RuntimeError: MPS backend out of memory` 가 나면 4 → 2로 낮추세요.
- `dataloader_num_workers`로 오디오 로딩을 병렬화합니다 (CPU 코어 활용).
- `save_steps`/`save_total_limit`을 조정해 체크포인트가 디스크를 다 채우지 않도록 합니다.


In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir=model_out_dir,
    num_train_epochs=3,
    per_device_train_batch_size=8,
    dataloader_num_workers=4,
    learning_rate=1e-5,
    save_steps=500,
    save_total_limit=3,
    logging_steps=20,
    predict_with_generate=True,
    fp16=False,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=collate_fn,
    processing_class=processor.feature_extractor,
)

print("학습 시작!")
trainer.train()
print("학습 완료!")


In [ ]:
model.save_pretrained(model_out_dir)
processor.save_pretrained(model_out_dir)
print(f"모델 저장 완료! -> {model_out_dir}")


In [ ]:
forced_decoder_ids = processor.get_decoder_prompt_ids(language="korean", task="transcribe")
model.config.forced_decoder_ids = forced_decoder_ids
model.to(device)

test_item = dataset_index[0]
speech, _ = librosa.load(test_item["audio_path"], sr=16000)
inputs = processor(speech, sampling_rate=16000, return_tensors="pt").to(device)

with torch.no_grad():
    predicted_ids = model.generate(inputs.input_features)
    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)

print("정답:", test_item["text"])
print("예측:", transcription[0])


In [10]:
print(next(model.parameters()).device)

mps:0
